How COCO Aligns with GIFs: The Strategy
The Key Insight
You're NOT training the LSTM on COCO images directly. Instead, you're training it to:

1.Learn the language of captions (grammar, sentence structure, common phrases)
2.Learn to generate text from visual features (the encoder-decoder pattern)
3.Then apply it to your GIF features

COCO Training:
Image Features → LSTM → "A person sitting on a bench"

Your GIF System:
[Emotion + Object + Action] Features → SAME LSTM → "A joyful person dancing energetically"

┌─────────────────────────────────────────────────────────┐
│                    TRAINING PHASE                        │
│  (Using COCO Images - teaches LSTM to write captions)   │
└─────────────────────────────────────────────────────────┘

COCO Image → ResNet50 Features (2048-dim)
                      ↓
              [Feature Vector]
                      ↓
                 LSTM Decoder
                      ↓
          "A person riding a bike"


┌─────────────────────────────────────────────────────────┐
│                   INFERENCE PHASE                        │
│      (Using YOUR GIFs - applies learned captioning)     │
└─────────────────────────────────────────────────────────┘

GIF → Emotion (512-dim) + Objects (embedding) + Action (embedding)
                      ↓
              [Feature Vector] ← Same size as COCO features!
                      ↓
            SAME LSTM Decoder (trained on COCO)
                      ↓
          "A joyful person dancing energetically"

Why This Works
1. Feature Space Alignment
Both COCO and GIFs produce feature vectors that go into the LSTM:

COCO: ResNet50 extracts 2048-dim features from images
GIFs: You combine emotion (512-dim) + object embeddings + action embeddings → project to 2048-dim

The LSTM doesn't care if features came from a static image or a GIF — it just sees a 2048-dimensional vector and generates text.
2. Transfer Learning
The LSTM learns:

✅ How to start captions ("A person...", "Someone is...", "An animated...")
✅ Grammar and syntax ("person is dancing" not "person dance")
✅ Common vocabulary (person, moving, happy, scene, etc.)
✅ How to end sentences properly

These skills transfer perfectly to GIFs!
3. Emotion Conditioning (Your Innovation)
Here's where you add your contribution:

# Standard COCO training
LSTM_input = [Image_Features]

# Your emotion-augmented version
LSTM_input = [Image_Features + Emotion_Vector]
```

You **inject emotion information** into the LSTM, forcing it to use emotion words. This is your unique contribution that makes GIF captions emotion-aware.

---

## **The Complete Training Pipeline**

### **Phase 1: Train Base LSTM on COCO (4-6 hours)**
```
COCO Image → ResNet Features → LSTM → Caption
Loss: Cross-Entropy (predict next word)
Result: LSTM that can write basic captions
```

### **Phase 2: Fine-tune with Emotion Conditioning (2-3 hours)**
```
COCO Image → [ResNet Features + Random Emotion Vector] → LSTM → Caption
Loss: Cross-Entropy + Emotion Word Penalty
Result: LSTM that includes emotion words in captions
```

### **Phase 3: Apply to Your GIFs (Inference, instant)**
```
GIF → [Emotion + Objects + Actions] → Trained LSTM → Caption
Result: "A joyful person dancing with a dog"
```

---

## **Why Not Train Directly on GIFs?**

You **could**, but:

❌ **Problem 1**: You only have ~5,000 GIFs, each with 1 caption (from templates)
- COCO has 120,000 images with **5 captions each** = 600,000 caption examples
- More data = better language model

❌ **Problem 2**: Your template captions are repetitive
- "A joyful person doing something"
- "A sad person moving around"
- Not enough variety for LSTM to learn rich language

✅ **Solution**: Train on COCO (rich captions) → Transfer to GIFs (emotion-aware)

---

## **Concrete Example**

Let's say LSTM trains on these COCO captions:
```
Image 1: "A woman sitting on a park bench reading a book"
Image 2: "A young boy playing with a soccer ball in a field"
Image 3: "Two people having a conversation at a cafe"
```

The LSTM learns:
- Sentence structure: "[Subject] [verb]ing [object] [location]"
- Vocabulary: person, sitting, playing, having, conversation
- Grammar: proper verb conjugation

**Then for your GIF:**
```
Input Features:
- Emotion: positive_energetic (high activation)
- Objects: [person, dog]
- Action: dancing

LSTM Output: "A joyful person dancing happily with a dog"

Your Specific Implementation
Here's how we'll actually do it:

# Step 1: Extract COCO image features (one-time preprocessing)
for image in COCO_dataset:
    features = resnet50(image)  # 2048-dim
    save(features)

# Step 2: Build vocabulary from COCO captions
vocab = build_vocab(all_coco_captions)  # ~10,000 words

# Step 3: Train LSTM
for epoch in range(20):
    for image_features, caption in COCO_data:
        # Add emotion embedding (randomly sampled during training)
        emotion_emb = random_emotion_embedding()  # Simulate emotion
        combined = concat([image_features, emotion_emb])
        
        # Train LSTM to predict caption word-by-word
        loss = LSTM.forward(combined, caption)
        LSTM.backward(loss)

# Step 4: Inference on YOUR GIFs
gif_features = extract_gif_features(gif)  # Emotion + Objects + Actions
caption = LSTM.generate(gif_features)

In [7]:
from pathlib import Path

# Check if files exist
files = ['train_features.pkl', 'val_features.pkl', 'vocab.pkl']

print("Checking for COCO features...")
for file in files:
    if Path(file).exists():
        size = Path(file).stat().st_size / (1024**2)  # Convert to MB
        print(f"✅ {file} found ({size:.1f} MB)")
    else:
        print(f"❌ {file} NOT FOUND - please place in current directory")

Checking for COCO features...
✅ train_features.pkl found (157.0 MB)
✅ val_features.pkl found (15.7 MB)
✅ vocab.pkl found (0.2 MB)


In [14]:
%pip install pycocotools nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# ============================================================
# LSTM CAPTION TRAINING - USING COCO FEATURES
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from tqdm import tqdm
import json
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')


print("=" * 60)
print("🚀 LSTM CAPTION TRAINING")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============================================================
# VOCABULARY CLASS DEFINITION (MUST BE DEFINED BEFORE LOADING)
# ============================================================

class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold
    
    def __len__(self):
        return len(self.itos)
    
    def build_vocabulary(self, captions):
        word_freq = Counter()
        idx = 4
        
        print("Tokenizing captions...")
        for caption in captions:
            tokens = word_tokenize(caption)
            word_freq.update(tokens)
        
        print(f"Unique tokens: {len(word_freq)}")
        
        for word, freq in word_freq.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
        
        print(f"Vocabulary size: {len(self.itos)}")
    
    def numericalize(self, caption):
        tokens = word_tokenize(caption.lower())
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokens]

# ============================================================
# LOAD COCO FEATURES & VOCABULARY
# ============================================================

print("\n📂 Loading COCO features and vocabulary...")

# Load vocabulary (now it will work!)
with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
print(f"✅ Vocabulary loaded: {len(vocab)} words")

# Load training features
with open('train_features.pkl', 'rb') as f:
    train_features = pickle.load(f)
print(f"✅ Training features loaded: {len(train_features)} images")

# Load validation features
with open('val_features.pkl', 'rb') as f:
    val_features = pickle.load(f)
print(f"✅ Validation features loaded: {len(val_features)} images")

print("\n" + "=" * 60)
print("✅ ALL DATA LOADED!")
print("=" * 60)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\akind\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\akind\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


🚀 LSTM CAPTION TRAINING
Device: cpu

📂 Loading COCO features and vocabulary...
✅ Vocabulary loaded: 10322 words
✅ Training features loaded: 20000 images
✅ Validation features loaded: 2000 images

✅ ALL DATA LOADED!


In [16]:
# ============================================================
# LOAD COCO CAPTIONS
# ============================================================
# We need the caption annotations (these are small JSON files)
# Download from: http://images.cocodataset.org/annotations/annotations_trainval2017.zip
# Extract and place the JSON files in current directory
# ============================================================

# If you don't have the annotations locally, download them:
import urllib.request
import zipfile
from pathlib import Path

annotations_path = Path('annotations')
if not annotations_path.exists():
    print("📥 Downloading COCO annotations...")
    url = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    urllib.request.urlretrieve(url, 'annotations.zip')
    
    with zipfile.ZipFile('annotations.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    print("✅ Annotations downloaded!")

# Load captions
import json

print("\n📖 Loading COCO captions...")

with open('annotations/captions_train2017.json', 'r') as f:
    train_coco = json.load(f)

with open('annotations/captions_val2017.json', 'r') as f:
    val_coco = json.load(f)

# Create image_id to captions mapping
train_captions = {}
for ann in train_coco['annotations']:
    img_id = ann['image_id']
    caption = ann['caption']
    
    if img_id not in train_captions:
        train_captions[img_id] = []
    train_captions[img_id].append(caption)

val_captions = {}
for ann in val_coco['annotations']:
    img_id = ann['image_id']
    caption = ann['caption']
    
    if img_id not in val_captions:
        val_captions[img_id] = []
    val_captions[img_id].append(caption)

# Filter to only images we have features for
train_captions = {k: v for k, v in train_captions.items() if k in train_features}
val_captions = {k: v for k, v in val_captions.items() if k in val_features}

print(f"✅ Training: {len(train_captions)} images with captions")
print(f"✅ Validation: {len(val_captions)} images with captions")

# Show sample
sample_id = list(train_captions.keys())[0]
print(f"\n📝 Sample captions for image {sample_id}:")
for i, cap in enumerate(train_captions[sample_id][:3]):
    print(f"  {i+1}. {cap}")


📖 Loading COCO captions...
✅ Training: 20000 images with captions
✅ Validation: 2000 images with captions

📝 Sample captions for image 16977:
  1. A car that seems to be parked illegally behind a legally parked car
  2. two cars parked on the sidewalk on the street
  3. City street with parked cars and a bench.


In [17]:
# ============================================================
# DATASET CLASS
# ============================================================

class COCOCaptionDataset(Dataset):
    def __init__(self, features, captions, vocab):
        """
        Args:
            features: dict {image_id: feature_vector}
            captions: dict {image_id: [caption1, caption2, ...]}
            vocab: Vocabulary object
        """
        self.features = features
        self.captions = captions
        self.vocab = vocab
        
        # Create list of (image_id, caption) pairs
        self.data = []
        for img_id, caps in captions.items():
            if img_id in features:
                for cap in caps:
                    self.data.append((img_id, cap))
        
        print(f"Dataset created: {len(self.data)} caption examples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_id, caption = self.data[idx]
        
        # Get features
        features = torch.FloatTensor(self.features[img_id])
        
        # Convert caption to indices
        caption_indices = [self.vocab.stoi["<SOS>"]]
        caption_indices += self.vocab.numericalize(caption)
        caption_indices.append(self.vocab.stoi["<EOS>"])
        
        caption_tensor = torch.LongTensor(caption_indices)
        
        return features, caption_tensor

def collate_fn(batch):
    """Pad captions to same length in batch"""
    features, captions = zip(*batch)
    
    features = torch.stack(features, 0)
    
    # Pad captions
    lengths = [len(cap) for cap in captions]
    max_len = max(lengths)
    
    padded_captions = torch.zeros(len(captions), max_len).long()
    for i, cap in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = cap
    
    return features, padded_captions, lengths

# Create datasets
print("\n📦 Creating datasets...")

train_dataset = COCOCaptionDataset(train_features, train_captions, vocab)
val_dataset = COCOCaptionDataset(val_features, val_captions, vocab)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print(f"✅ Training batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")


📦 Creating datasets...
Dataset created: 100056 caption examples
Dataset created: 10007 caption examples
✅ Training batches: 1564
✅ Validation batches: 157


In [18]:
# ============================================================
# LSTM CAPTION DECODER
# ============================================================

class CaptionLSTM(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=2):
        super(CaptionLSTM, self).__init__()
        
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        
        # Project image features to embedding space
        self.feature_fc = nn.Linear(feature_dim, embed_dim)
        
        # Word embeddings
        self.word_embed = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        
        # Output layer
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, features, captions):
        """
        Args:
            features: (batch, 2048)
            captions: (batch, seq_len)
        Returns:
            outputs: (batch, seq_len, vocab_size)
        """
        # Project features
        features_embedded = self.feature_fc(features).unsqueeze(1)
        
        # Embed captions (exclude last token for teacher forcing)
        caption_embedded = self.word_embed(captions[:, :-1])
        
        # Concatenate
        embeddings = torch.cat([features_embedded, caption_embedded], dim=1)
        
        # LSTM
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = self.dropout(lstm_out)
        
        # Predict words
        outputs = self.fc_out(lstm_out)
        
        return outputs
    
    def generate_caption(self, features, max_len=20):
        """Generate caption (inference)"""
        self.eval()
        
        with torch.no_grad():
            caption = [vocab.stoi["<SOS>"]]
            features_embedded = self.feature_fc(features).unsqueeze(1)
            hidden = None
            
            for _ in range(max_len):
                word_tensor = torch.LongTensor([caption[-1]]).to(features.device)
                word_embedded = self.word_embed(word_tensor).unsqueeze(1)
                
                if len(caption) == 1:
                    lstm_input = features_embedded
                else:
                    lstm_input = word_embedded
                
                lstm_out, hidden = self.lstm(lstm_input, hidden)
                output = self.fc_out(lstm_out.squeeze(1))
                predicted = output.argmax(1).item()
                
                caption.append(predicted)
                
                if predicted == vocab.stoi["<EOS>"]:
                    break
            
            # Convert to words
            caption_words = [vocab.itos[idx] for idx in caption[1:-1]]
            return ' '.join(caption_words)

# Initialize model
model = CaptionLSTM(
    feature_dim=2048,
    embed_dim=256,
    hidden_dim=512,
    vocab_size=len(vocab),
    num_layers=2
).to(device)

print("=" * 60)
print("🧠 LSTM MODEL INITIALIZED")
print("=" * 60)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size: {len(vocab)}")

🧠 LSTM MODEL INITIALIZED
Parameters: 12,140,370
Vocabulary size: 10322


The problem is in how the model outputs are created. The issue is that the model concatenates image features with caption embeddings, making the output sequence 1 token longer than expected. Let me check what the debug output shows before the error. Can you share the "Debug shapes" output that was printed before the error occurred?
But actually, I can see the pattern - the mismatch is always 64 (one batch size). This means the output has one extra token. Let me fix the model's forward pass

In [25]:
# ============================================================
# LSTM CAPTION DECODER (FIXED)
# ============================================================

class CaptionLSTM(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=2):
        super(CaptionLSTM, self).__init__()
        
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        
        # Project image features to embedding space
        self.feature_fc = nn.Linear(feature_dim, embed_dim)
        
        # Word embeddings
        self.word_embed = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        
        # Output layer
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, features, captions):
        """
        Args:
            features: (batch, 2048)
            captions: (batch, seq_len) - includes <SOS> and <EOS>
        Returns:
            outputs: (batch, seq_len-1, vocab_size) - predictions for each word after <SOS>
        """
        # Embed caption words (exclude last token)
        # If caption is [<SOS>, w1, w2, w3, <EOS>]
        # We embed [<SOS>, w1, w2, w3] and predict [w1, w2, w3, <EOS>]
        caption_embedded = self.word_embed(captions[:, :-1])  # (batch, seq_len-1, embed_dim)
        
        # Project features and prepend to caption embeddings
        features_embedded = self.feature_fc(features).unsqueeze(1)  # (batch, 1, embed_dim)
        
        # Concatenate: [features, <SOS>, w1, w2, ...]
        embeddings = torch.cat([features_embedded, caption_embedded], dim=1)  # (batch, seq_len, embed_dim)
        
        # Pass through LSTM
        lstm_out, _ = self.lstm(embeddings)  # (batch, seq_len, hidden_dim)
        
        # We want to predict from the features + words, so skip the feature embedding output
        lstm_out = lstm_out[:, 1:, :]  # (batch, seq_len-1, hidden_dim)
        
        # Dropout
        lstm_out = self.dropout(lstm_out)
        
        # Predict words
        outputs = self.fc_out(lstm_out)  # (batch, seq_len-1, vocab_size)
        
        return outputs
    
    def generate_caption(self, features, max_len=20):
        """Generate caption (inference)"""
        self.eval()
        
        with torch.no_grad():
            caption = [vocab.stoi["<SOS>"]]
            features_embedded = self.feature_fc(features).unsqueeze(1)
            hidden = None
            
            for _ in range(max_len):
                word_tensor = torch.LongTensor([caption[-1]]).to(features.device)
                word_embedded = self.word_embed(word_tensor).unsqueeze(1)
                
                if len(caption) == 1:
                    lstm_input = features_embedded
                else:
                    lstm_input = word_embedded
                
                lstm_out, hidden = self.lstm(lstm_input, hidden)
                output = self.fc_out(lstm_out.squeeze(1))
                predicted = output.argmax(1).item()
                
                caption.append(predicted)
                
                if predicted == vocab.stoi["<EOS>"]:
                    break
            
            # Convert to words
            caption_words = [vocab.itos[idx] for idx in caption[1:-1]]
            return ' '.join(caption_words)

# Re-initialize model with fixed forward pass
model = CaptionLSTM(
    feature_dim=2048,
    embed_dim=256,
    hidden_dim=512,
    vocab_size=len(vocab),
    num_layers=2
).to(device)

print("=" * 60)
print("🧠 LSTM MODEL INITIALIZED (FIXED)")
print("=" * 60)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size: {len(vocab)}")

🧠 LSTM MODEL INITIALIZED (FIXED)
Parameters: 12,140,370
Vocabulary size: 10322


In [26]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
best_val_loss = float('inf')

print("=" * 60)
print("🏋️ TRAINING LSTM")
print("=" * 60)
print(f"Epochs: {num_epochs}")
print(f"Batch size: {batch_size}")
print(f"Device: {device}")
print()

# ============================================================
# TRAINING LOOP (SIMPLIFIED - NOW SHAPES MATCH!)
# ============================================================

for epoch in range(num_epochs):
    print(f"📍 Epoch {epoch+1}/{num_epochs}")
    print("-" * 60)
    
    # Training
    model.train()
    train_loss = 0
    
    for features, captions, lengths in tqdm(train_loader, desc="Training"):
        features = features.to(device)
        captions = captions.to(device)
        
        # Forward
        outputs = model(features, captions)
        # outputs: (batch, seq_len-1, vocab_size)
        # captions[:, 1:]: (batch, seq_len-1) - target words (skip <SOS>)
        
        # Reshape for loss
        outputs_flat = outputs.reshape(-1, len(vocab))
        targets_flat = captions[:, 1:].reshape(-1)
        
        # Calculate loss
        loss = criterion(outputs_flat, targets_flat)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for features, captions, lengths in tqdm(val_loader, desc="Validation"):
            features = features.to(device)
            captions = captions.to(device)
            
            outputs = model(features, captions)
            outputs_flat = outputs.reshape(-1, len(vocab))
            targets_flat = captions[:, 1:].reshape(-1)
            
            loss = criterion(outputs_flat, targets_flat)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    
    # Print stats
    print(f"📊 Epoch {epoch+1} Summary:")
    print(f"   Train Loss: {avg_train_loss:.4f}")
    print(f"   Val Loss:   {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_caption_model.pth')
        print(f"   ✅ Best model saved! (Val Loss: {avg_val_loss:.4f})")
    
    # Generate sample captions
    print(f"   💬 Sample captions:")
    model.eval()
    for i in range(3):
        sample_features = next(iter(val_loader))[0][i].unsqueeze(0).to(device)
        sample_caption = model.generate_caption(sample_features)
        print(f"      {i+1}. {sample_caption}")
    print()

print("=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Model saved as: best_caption_model.pth")

🏋️ TRAINING LSTM
Epochs: 10
Batch size: 64
Device: cpu

📍 Epoch 1/10
------------------------------------------------------------


Validation: 100%|██████████| 157/157 [00:31<00:00,  4.95it/s]


📊 Epoch 1 Summary:
   Train Loss: 3.7510
   Val Loss:   3.0758
   ✅ Best model saved! (Val Loss: 3.0758)
   💬 Sample captions:
      1. a man is standing in the snow .
      2. a man is standing in the snow .
      3. a man is standing in the snow .

📍 Epoch 2/10
------------------------------------------------------------


Validation: 100%|██████████| 157/157 [00:31<00:00,  4.94it/s]


📊 Epoch 2 Summary:
   Train Loss: 2.9699
   Val Loss:   2.7683
   ✅ Best model saved! (Val Loss: 2.7683)
   💬 Sample captions:
      1. a man is riding a skateboard down a street .
      2. a man is riding a skateboard down a street .
      3. a man is riding a skateboard down a street .

📍 Epoch 3/10
------------------------------------------------------------


...

📊 Epoch 8 Summary:
   Train Loss: 2.2279
   Val Loss:   2.5766
   💬 Sample captions:
      1. gates a motorcycle parked on the side of the road .
      2. gates a motorcycle parked on the side of the road .
      3. gates a motorcycle parked on the side of the road .

📍 Epoch 9/10
------------------------------------------------------------


Validation: 100%|██████████| 157/157 [00:31<00:00,  5.00it/s]


📊 Epoch 9 Summary:
   Train Loss: 2.1654
   Val Loss:   2.5851
   💬 Sample captions:
      1. gates a motorcycle is parked on the side of the road .
      2. gates a motorcycle is parked on the side of the road .
      3. gates a motorcycle is parked on the side of the road .

📍 Epoch 10/10
------------------------------------------------------------


Validation: 100%|██████████| 157/157 [00:31<00:00,  5.06it/s]


📊 Epoch 10 Summary:
   Train Loss: 2.1072
   Val Loss:   2.5994
   💬 Sample captions:
      1. gates a motorcycle parked on a street .
      2. gates a motorcycle parked on a street .
      3. gates a motorcycle parked on a street .

✅ TRAINING COMPLETE!
Best validation loss: 2.5584
Model saved as: best_caption_model.pth
